# LLM Streaming — Token-by-Token Output

Instead of waiting for the full response, **streaming** lets you display tokens as they are generated — critical for responsive UIs and long outputs.

Each provider uses a slightly different streaming API, but the pattern is the same: open a stream, iterate chunks, print incrementally.

In [ ]:
import os

PROMPT = 'Write a short poem about machine learning in exactly four lines.'

---
## Ollama (local)

Set `stream=True`; each chunk is a dict; read `chunk['message']['content']`.

In [ ]:
import ollama

OLLAMA_MODEL = 'mistral-nemo:12b-instruct-2407-q4_K_M'

for chunk in ollama.chat(
    model=OLLAMA_MODEL,
    messages=[{'role': 'user', 'content': PROMPT}],
    stream=True
):
    print(chunk['message']['content'], end='', flush=True)
print()

---
## OpenAI

Set `stream=True`; iterate over chunks and read `.choices[0].delta.content`.

In [ ]:
from openai import OpenAI

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
OPENAI_MODEL   = 'gpt-5'

openai_client = OpenAI(api_key=OPENAI_API_KEY)
stream = openai_client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[{'role': 'user', 'content': PROMPT}],
    stream=True
)
for chunk in stream:
    token = chunk.choices[0].delta.content
    if token:
        print(token, end='', flush=True)
print()

---
## Anthropic

Use `client.messages.stream()` as a context manager; iterate `.text_stream` for token strings.

In [ ]:
from anthropic import Anthropic

ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
ANTHROPIC_MODEL   = 'claude-sonnet-4-5-20250929'

anthropic_client = Anthropic(api_key=ANTHROPIC_API_KEY)
with anthropic_client.messages.stream(
    model=ANTHROPIC_MODEL,
    max_tokens=300,
    messages=[{'role': 'user', 'content': PROMPT}]
) as stream:
    for token in stream.text_stream:
        print(token, end='', flush=True)
print()

---
## Google Gemini

Pass `stream=True` to `generate_content()`; each chunk has a `.text` attribute.

In [ ]:
import google.generativeai as genai

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
GOOGLE_MODEL   = 'gemini-2.5-flash'

genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel(GOOGLE_MODEL)
for chunk in gemini_model.generate_content(PROMPT, stream=True):
    print(chunk.text, end='', flush=True)
print()